# Treino YOLO com os datasets gerados pela Unity

Usa as pastas `run_<data>_seed<N>` criadas pelo gerador de dataset (`images/`, `labels/`, `data.yaml`).

**Preparar o ambiente (uma vez)**, no terminal do VS Code, dentro desta pasta:

```powershell
py -m venv .venv
.venv\Scripts\Activate.ps1
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124   # só com GPU NVIDIA
pip install -r requirements.txt
```

Depois, no canto superior direito do notebook, escolha o kernel **.venv** e rode as células em ordem.

## 1. Configuração

In [ ]:
from pathlib import Path

# Onde a Unity grava os datasets (pastas run_*)
DATASET_ROOT = Path(r"C:\Users\felip\Documents\Mestrado\Insetos\dataset")

# Execuções a usar. Vazio = as ULTIMAS mais recentes dentro de DATASET_ROOT.
RUNS = [
    # r"C:\Users\felip\Documents\Mestrado\Insetos\dataset\run_20261110_010713_seed1948904496",
]
ULTIMAS = 1

# Treino
MODEL = "yolo11n.pt"   # yolo11n/s/m.pt, yolov8n.pt... ou um .yaml para treinar do zero
EPOCHS = 100
IMGSZ = 512            # as imagens da Unity têm 512x512
BATCH = 16             # -1 = automático pelo tamanho da GPU
DEVICE = None          # None = detecta (GPU 0 se houver), ou 0, ou "cpu"
WORKERS = 0            # 0 é o mais seguro no Jupyter/Windows; tente 4 se não travar
PATIENCE = 30          # para cedo se a validação não melhorar
SEED = 0
NOME = None            # nome da pasta do treino em runs/ (None = treino_<data>)

# Avaliação em fotos reais anotadas (pasta com images/ e labels/). None = pular.
PASTA_REAL = None

PASTA_SAIDA = Path.cwd() / "runs"

## 2. Funções auxiliares

In [ ]:
import sys
from datetime import datetime

EXT_IMG = {".png", ".jpg", ".jpeg", ".bmp"}


def ler_nomes_classes(data_yaml: Path):
    # Lê o bloco 'names:' do data.yaml escrito pela Unity (sem depender de PyYAML)
    nomes, dentro = {}, False
    for linha in data_yaml.read_text(encoding="utf-8").splitlines():
        if linha.strip().startswith("names:"):
            dentro = True
            continue
        if dentro:
            if not linha.startswith(" "):
                break
            chave, _, valor = linha.strip().partition(":")
            if chave.strip().isdigit():
                nomes[int(chave)] = valor.strip()
    return [nomes[i] for i in sorted(nomes)]


def verificar_split(run: Path, split: str, n_classes: int):
    # Confere pares imagem/label e o formato de cada label. Retorna (n_imagens, problemas)
    pasta_img, pasta_lbl = run / "images" / split, run / "labels" / split
    problemas = []
    if not pasta_img.exists():
        return 0, problemas
    imagens = [p for p in pasta_img.iterdir() if p.suffix.lower() in EXT_IMG]
    for img in imagens:
        lbl = pasta_lbl / (img.stem + ".txt")
        if not lbl.exists():
            problemas.append(f"sem label: {img.name}")
            continue
        for n, linha in enumerate(lbl.read_text(encoding="utf-8").splitlines(), 1):
            if not linha.strip():
                continue
            partes = linha.split()
            if len(partes) != 5:
                problemas.append(f"{lbl.name}:{n} não tem 5 colunas: '{linha}'")
                continue
            try:
                cls, vals = int(partes[0]), [float(v) for v in partes[1:]]
            except ValueError:
                problemas.append(f"{lbl.name}:{n} número inválido (vírgula decimal?): '{linha}'")
                continue
            if not 0 <= cls < n_classes:
                problemas.append(f"{lbl.name}:{n} classe {cls} fora de [0, {n_classes - 1}]")
            if any(v != v for v in vals) or any(v < 0 or v > 1 for v in vals):
                problemas.append(f"{lbl.name}:{n} coordenada fora de [0, 1] ou NaN: '{linha}'")
            elif vals[2] <= 0 or vals[3] <= 0:
                problemas.append(f"{lbl.name}:{n} largura/altura zero")
    if pasta_lbl.exists():
        stems = {p.stem for p in imagens}
        problemas += [f"label sem imagem: {l.name}" for l in pasta_lbl.glob("*.txt") if l.stem not in stems]
    return len(imagens), problemas


def verificar_runs(runs):
    nomes_ref, total_train, total_val, todos = None, 0, 0, []
    for run in runs:
        data_yaml = run / "data.yaml"
        if not data_yaml.exists():
            raise FileNotFoundError(f"{run} não tem data.yaml (não é uma pasta run_* gerada pela Unity?)")
        nomes = ler_nomes_classes(data_yaml)
        if nomes_ref is None:
            nomes_ref = nomes
        elif nomes != nomes_ref:
            raise ValueError(f"As classes de {run.name} ({nomes}) diferem das anteriores ({nomes_ref}).")
        n_tr, p_tr = verificar_split(run, "train", len(nomes))
        n_va, p_va = verificar_split(run, "val", len(nomes))
        total_train += n_tr
        total_val += n_va
        todos += [f"[{run.name}/train] {p}" for p in p_tr] + [f"[{run.name}/val] {p}" for p in p_va]
        print(f"  {run.name}: {n_tr} treino, {n_va} validação, {len(p_tr) + len(p_va)} problema(s)")
    print(f"\nClasses: {nomes_ref}")
    print(f"Total: {total_train} imagens de treino, {total_val} de validação")
    if todos:
        print(f"\n[ATENÇÃO] {len(todos)} problema(s). O YOLO descarta os labels com defeito. Primeiros 20:")
        for p in todos[:20]:
            print("   ", p)
    else:
        print("[OK] Todos os labels estão no formato YOLO correto.")
    return nomes_ref, total_train, total_val, todos


def escrever_data_yaml(destino: Path, runs, nomes, usar_train_como_val: bool):
    # data.yaml combinado com caminhos absolutos (o Ultralytics aceita listas de pastas)
    def lista(split):
        return [str((r / "images" / split).resolve()).replace("\\", "/") for r in runs
                if (r / "images" / split).exists() and any((r / "images" / split).iterdir())]
    train = lista("train")
    val = train if usar_train_como_val else lista("val")
    linhas = ["train:"] + [f"  - {p}" for p in train] + ["val:"] + [f"  - {p}" for p in val]
    linhas += ["names:"] + [f"  {i}: {n}" for i, n in enumerate(nomes)]
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text("\n".join(linhas) + "\n", encoding="utf-8")
    return destino


def escrever_yaml_real(destino: Path, pasta_real: Path, nomes):
    img = pasta_real / "images"
    if not img.exists():
        raise FileNotFoundError(f"{pasta_real} precisa ter as subpastas images/ e labels/")
    p = str(img.resolve()).replace("\\", "/")
    linhas = [f"train: {p}", f"val: {p}", "names:"] + [f"  {i}: {n}" for i, n in enumerate(nomes)]
    destino.write_text("\n".join(linhas) + "\n", encoding="utf-8")
    return destino

## 3. Escolher as execuções e conferir o dataset

In [ ]:
if RUNS:
    runs = [Path(r) for r in RUNS]
else:
    candidatos = sorted(DATASET_ROOT.glob("run_*"), key=lambda p: p.stat().st_mtime) if DATASET_ROOT.exists() else []
    if not candidatos:
        raise FileNotFoundError(f"Nenhuma pasta run_* em {DATASET_ROOT}. Gere um dataset na Unity ou preencha RUNS.")
    runs = candidatos[-ULTIMAS:]
for r in runs:
    if not r.is_dir():
        raise FileNotFoundError(f"Pasta não encontrada: {r}")

print("Execuções usadas:")
nomes, n_train, n_val, problemas = verificar_runs(runs)
assert n_train > 0, "Nenhuma imagem de treino."

Visualização rápida de algumas imagens com as caixas dos labels (antes de treinar):

In [ ]:
import random
from PIL import Image, ImageDraw
from IPython.display import display

def desenhar_label(img_path: Path):
    lbl = img_path.parent.parent.parent / "labels" / img_path.parent.name / (img_path.stem + ".txt")
    im = Image.open(img_path).convert("RGB")
    d = ImageDraw.Draw(im)
    W, H = im.size
    if lbl.exists():
        for linha in lbl.read_text().splitlines():
            p = linha.split()
            if len(p) == 5:
                try:
                    c, xc, yc, w, h = int(p[0]), *map(float, p[1:])
                except ValueError:
                    continue  # label com defeito (já listado na conferência acima)
                d.rectangle([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], outline=(0, 255, 0), width=2)
                d.text(((xc - w / 2) * W + 3, (yc - h / 2) * H + 2), nomes[c] if c < len(nomes) else str(c), fill=(0, 255, 0))
    return im

amostras = [p for r in runs for p in (r / "images" / "train").glob("*.png")]
amostras = random.sample(amostras, min(8, len(amostras)))
if amostras:
    thumbs = [desenhar_label(p).resize((256, 256)) for p in amostras]
    grade = Image.new("RGB", (256 * min(4, len(thumbs)), 256 * ((len(thumbs) + 3) // 4)))
    for k, t in enumerate(thumbs):
        grade.paste(t, ((k % 4) * 256, (k // 4) * 256))
    display(grade)

## 4. Montar o data.yaml combinado

In [ ]:
usar_train_como_val = n_val == 0
if usar_train_como_val:
    print("[ATENÇÃO] Nenhuma imagem de validação: usando o treino como validação "
          "(métricas otimistas). Aumente valFraction na Unity.")

nome = NOME or f"treino_{datetime.now():%Y%m%d_%H%M%S}"
pasta_treino = PASTA_SAIDA / nome
data_yaml = escrever_data_yaml(pasta_treino / "data_combinado.yaml", runs, nomes, usar_train_como_val)
(pasta_treino / "runs_usadas.txt").write_text("\n".join(str(r) for r in runs) + "\n", encoding="utf-8")
print(data_yaml.read_text())

## 5. Treinar

In [ ]:
from ultralytics import YOLO

device = DEVICE
if device is None:
    import torch
    device = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", f"GPU {device}" if device != "cpu" else "CPU (vai ser lento)")

model = YOLO(MODEL)
resultado = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    workers=WORKERS,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    project=str(PASTA_SAIDA),
    name=nome,
    exist_ok=True,
    plots=True,
)
best = pasta_treino / "weights" / "best.pt"
print("Melhor modelo:", best)

## 6. Resultados

In [ ]:
from IPython.display import Image as IPImage, display

for arq in ["results.png", "labels.jpg", "train_batch0.jpg", "val_batch0_labels.jpg", "val_batch0_pred.jpg", "confusion_matrix.png"]:
    p = pasta_treino / arq
    if p.exists():
        print(arq)
        display(IPImage(filename=str(p), width=800))

## 7. Avaliação em fotos reais (opcional)
Preencha `PASTA_REAL` na configuração (pasta com `images/` e `labels/`).

In [ ]:
if PASTA_REAL:
    yaml_real = escrever_yaml_real(pasta_treino / "data_real.yaml", Path(PASTA_REAL), nomes)
    metricas = YOLO(str(best)).val(data=str(yaml_real), imgsz=IMGSZ, device=device,
                                   project=str(PASTA_SAIDA), name=nome + "_real", exist_ok=True)
    print(f"mAP50 real: {metricas.box.map50:.4f} | mAP50-95 real: {metricas.box.map:.4f}")
else:
    print("PASTA_REAL não definida: pulando.")

## 8. Testar o modelo em imagens quaisquer (opcional)

In [ ]:
IMAGENS_TESTE = [
    # r"C:\caminho\foto1.jpg",
]
if IMAGENS_TESTE:
    for r in YOLO(str(best)).predict(IMAGENS_TESTE, imgsz=IMGSZ, device=device, conf=0.25):
        display(Image.fromarray(r.plot()[..., ::-1]))